In [67]:
import pandas as pd

In [68]:
df = pd.read_csv("../../model_2/data/raw/sofifa_players.csv", index_col=False)

/var/folders/0_/_3dg0b615p38ptjrygv5db6h0000gn/T/ipykernel_86457/2871305303.py:1: DtypeWarning: Columns (3,4,13,14,19,20,21,22,23,25,26,27,28,29,31,32,33,34,35,37,38,39,40,41,43,44,45,46,47,48,50,51,52,57,58,69,70,71,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../model_2/data/raw/sofifa_players.csv", index_col=False)


In [69]:
df = df[df['season'] == 2023].drop(["Unnamed: 0", 'Traits', 'Traits.1',
       'PlayStyles', 'PlayStyles +',  'Unnamed: 76', "Acceleration type", "Loan date end"], axis=1)

In [ ]:
import re

def _clean_sofifa_data( df):
    df.columns = df.columns.str.replace(" / ", "_").str.replace(" & ", "_").str.replace(" ", "_")

    df['height(cm)'] = df["height"].str.split("cm").str[0].astype(int)
    df['weight(kg)'] = df["weight"].str.split("kg").str[0].astype(int) 

    df['foot'] = df['foot'].map({"Left": 1, "Right": 2})

def parse_monetary_value(row): 
    row = row.split('€')[1]
    if "M" in row:
        value = float(row.replace("M", "")) * 1_000_000
    elif "K" in row:
        value = float(row.replace("K", "")) * 1_000
    else:
        value = float(row)

    return value

def extract_team_name(row):
    """Extract team name from 'Team & Contract' column by splitting on first 4-digit pattern"""
    match = re.search(r'\d{4}', str(row))
    if match:
        # Get position of first 4-digit pattern
        position = match.start()
        # Extract everything before it and strip whitespace
        team_name = row[:position].strip()
        return team_name
    else:
        # If no 4-digit pattern found, return original value
        return row

def extract_primary_position(row):
    if "," in row:
        row = row.split(",")[0]
    
    return row.strip()

def extract_name_from_position(row):
    pattern = r'[A-Z]{2,}'
    match = re.search(pattern, row)
    if match:
        location = match.start()
        name = row[:location].strip()
        
        return name
    else:
        return "HELLO"

df.drop_duplicates(inplace=True)
df.columns = df.columns.str.strip().str.lower()

df['name'] = df['name'].apply(extract_name_from_position)
df["name"] = df['name'].str.lower()      

_clean_sofifa_data(df)
df['value(€)'] = df['value'].apply(parse_monetary_value)
df['wage(€)'] = df['wage'].apply(parse_monetary_value)
df['team'] = df['team_contract'].apply(extract_team_name)
df.drop(['team_contract', 'height', 'weight', 'real_face', 'body_type', 'value', 'wage'], axis=1, inplace=True)


In [73]:
# max column number
pd.set_option('display.max_columns', None)